***Получение прогноза для тестовых данных***

Распакуем наши обработчики

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mamoUtils as mu
np.__version__, pd.__version__

('2.5.5', '3.1.1')

In [25]:
import joblib

pipeline = joblib.load('models/best_model.pkl')

bestForest_1 = pipeline['model_1']
bestForest_2 = pipeline['model_2']
bestForest_3 = pipeline['model_3']
yearsScaler = pipeline['years_scaler']
monthEncoder = pipeline['month_encoder']
unitEncoder = pipeline['unit_encoder']
skuEncoder = pipeline['sku_encoder']
scaler_1 = pipeline['param_scaler_1']
scaler_2 = pipeline['param_scaler_2']
scaler_3 = pipeline['param_scaler_3']

Далее загружаем из файла данные для прогноза

In [26]:
testDataFrame = pd.read_csv('data/test.csv')
testDataFrame.head()

,id,sku,unit,month
0,0_2021-12,0,ШТ,2021-12
1,0_2022-01,0,ШТ,2022-01
2,0_2022-02,0,ШТ,2022-02
3,1_2021-12,1,ШТ,2021-12
4,1_2022-01,1,ШТ,2022-01


In [27]:
fillDataFrame = pd.read_csv('data/train.csv')
fillDataFrame.head()

,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,379.0,250.0,317.0,172.0,193.0,173.0,121.0,318.0,328.0,208.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,68.0,65.0,72.0,82.0,85.0,69.0,75.0,100.0,72.0,89.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,334.0,380.0,409.0,309.0,388.0,438.0,404.0,293.0,274.0,381.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0,2187.0,1672.0,1665.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,222.0,258.0,289.0,662.0,759.0,616.0,365.0,321.0,392.0,409.0


In [28]:
splitter = mu.SplitDateOnColumn()
splittedData = splitter.transform(fillDataFrame)
dateSplitter = mu.SplitDateOnYearsMonth()
transformedFillData = dateSplitter.transform(splittedData)
transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue'] = 0
transformedFillData['resultValue'] = np.log(transformedFillData['resultValue'] + 1)
print(len(transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue']))
transformedFillData.head(40)

0


,sku,unit,resultValue,years,month
0,0,ШТ,4.859812,2019,1
1,0,ШТ,4.836282,2019,2
2,0,ШТ,5.192957,2019,3
3,0,ШТ,5.298317,2019,4
4,0,ШТ,5.323010,2019,5
5,0,ШТ,5.509388,2019,6
6,0,ШТ,5.153292,2019,7
7,0,ШТ,5.463832,2019,8
8,0,ШТ,5.476464,2019,9
9,0,ШТ,5.433722,2019,10


In [29]:
testDataFrameDates = dateSplitter.transform(testDataFrame)
testDataFrameDates.head()

,id,sku,unit,years,month
0,0_2021-12,0,ШТ,2021,12
1,0_2022-01,0,ШТ,2022,1
2,0_2022-02,0,ШТ,2022,2
3,1_2021-12,1,ШТ,2021,12
4,1_2022-01,1,ШТ,2022,1


In [30]:
ids = testDataFrameDates[['id']]
testDataFrameDates = testDataFrameDates.drop('id', axis = 1)
OneMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2021) & (testDataFrameDates['month'] == 12)]
TwoMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 1)]
ThreeMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 2)]

In [31]:
def get_lag_for_date(dateFrame, lagsFrom, lags):
    df = dateFrame.copy()
    columns = []
    for i in lags:
        lag_string = f'lag_{i}'
        columns += [lag_string]
        targetYear = dateFrame.iloc[0]['years']
        targetMonth = dateFrame.iloc[0]['month'] - i;
        while(targetMonth <= 0):
            targetYear -= 1
            targetMonth += 12
        lagVal = lagsFrom.loc[(lagsFrom['years'] == targetYear) & (lagsFrom['month'] == targetMonth), ['sku','resultValue']]
        lagVal = lagVal.rename(columns={'resultValue': lag_string})
        df = df.merge(lagVal, on='sku', how='left')
    return (df, columns)
def get_rolling_mean_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rm_str = f'rolling_mean_{i}'
        columns += [rm_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        mean = resValForMean.groupby('sku')[['resultValue']].mean()
        mean = mean.rename(columns={'resultValue': rm_str})
        df = df.merge(mean, on='sku', how='left')
    return (df, columns)
def get_rolling_std_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rs_str = f'rolling_std_{i}'
        columns += [rs_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        std = resValForMean.groupby('sku')[['resultValue']].std()
        std = std.rename(columns={'resultValue': rs_str})
        df = df.merge(std, on='sku', how='left')
    return (df, columns)
def get_season_ratio(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_ratio'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] / (df[season_str] + 1);
    return (df, [season_str])
def get_season_deviation(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_deviation'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] - df[season_str];
    return (df, [season_str])
def get_dataFrame_with_params(dataFrame, fillDataFrame, lags, yearsFromWindows, monthFromWindows, windows):
    baseIndexes = dataFrame.index
    df = dataFrame.copy()
    df, lag_columns = get_lag_for_date(df, fillDataFrame, lags)
    df, mean_columns = get_rolling_mean_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, std_columns = get_rolling_std_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, season_columns = get_season_ratio(df, fillDataFrame, lag_columns[0])
    df, season_dev_columns = get_season_deviation(df, fillDataFrame, lag_columns[0])
    df.index = baseIndexes
    return (df, lag_columns + mean_columns + std_columns + season_dev_columns, season_columns)

data_1, columns_1, freeColumn_1 = get_dataFrame_with_params(OneMonthDataFrame, transformedFillData, [1, 2, 3, 6, 12], 2021, 11, [3, 4, 6, 12])
data_1.head()

,sku,unit,years,month,lag_1,lag_2,lag_3,lag_6,lag_12,rolling_mean_3,rolling_mean_4,rolling_mean_6,rolling_mean_12,rolling_std_3,rolling_std_4,rolling_std_6,rolling_std_12,season_ratio,season_deviation
0,0,ШТ,2021,12,5.342334,5.796058,5.765191,5.267858,5.605802,5.634528,5.426901,5.355753,5.494867,0.253517,0.463986,0.377495,0.347990,0.842161,-0.001269
3,1,ШТ,2021,12,4.499810,4.290459,4.615121,4.454347,4.394449,4.468463,4.434031,4.406494,4.319806,0.164585,0.151000,0.140492,0.187216,0.836093,0.117861
6,2,ШТ,2021,12,5.945421,5.616771,5.683580,5.963579,5.793014,5.74859,5.812415,5.882956,5.867629,0.173702,0.190812,0.187751,0.145904,0.911140,0.420166
9,3,ШТ,2021,12,7.418181,7.422374,7.690743,7.962764,7.393878,7.510433,7.509747,7.616706,7.646081,0.156168,0.127518,0.210232,0.194503,0.863876,-0.168910
12,4,ШТ,2021,12,6.016157,5.973810,5.774552,6.633318,6.396930,5.921506,5.916788,6.12089,5.9713,0.129016,0.105763,0.333219,0.431797,0.830230,-0.230220


In [32]:
def scale_data(dataFrame, scaledColumns, freeColumns, yearsScaler, paramScaler, monthEncoder, unitEncoder, skuEncoder):
    encoded_data = encode_data(dataFrame, unitEncoder, skuEncoder)
    scaledDate = pd.DataFrame(yearsScaler.transform(dataFrame[['years']]), index = dataFrame.index, columns = ['years'])
    encodedMonth = monthEncoder.transform(dataFrame[['month']]).toarray()
    encodedMonthDF = pd.DataFrame(encodedMonth, index = dataFrame.index, columns = monthEncoder.get_feature_names_out(input_features=dataFrame[['month']].columns))
    scaledParam = pd.DataFrame(
        paramScaler.transform(
            encoded_data[['sku']].join(
                dataFrame[scaledColumns]
            )
        ), index = dataFrame.index,
        columns = ['sku'] + scaledColumns)
    resultDataFrame = scaledParam.join(encoded_data.drop('sku', axis=1)).join(encodedMonthDF).join(scaledDate).join(dataFrame[freeColumns])
    return resultDataFrame

def encode_data(dataFrame, unitEncoder, skuEncoder):
    encodedSku = pd.DataFrame(skuEncoder.transform(dataFrame[['sku']].astype(str)), index = dataFrame.index, columns = ['sku'])
    encodedUnit = unitEncoder.transform(dataFrame[['unit']]).toarray()
    encodedUnitDF = pd.DataFrame(encodedUnit, index = dataFrame.index, columns = unitEncoder.get_feature_names_out(input_features=dataFrame[['unit']].columns))
    return encodedSku.join(encodedUnitDF)

scaled_X_test_1 = scale_data(data_1, columns_1, freeColumn_1, yearsScaler, scaler_1, monthEncoder, unitEncoder, skuEncoder)

In [33]:
scaled_X_test_1.head()

,sku,lag_1,lag_2,lag_3,lag_6,lag_12,rolling_mean_3,rolling_mean_4,rolling_mean_6,rolling_mean_12,...,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,years,season_ratio
0,-0.167714,-0.157269,0.078927,0.066997,-0.199004,0.027336,0.008124,-0.093419,-0.132812,-0.055004,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.842161
3,-0.614283,-0.579713,-0.674590,-0.508103,-0.604731,-0.556403,-0.587949,-0.604353,-0.629366,-0.690094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.836093
6,0.039391,0.145120,-0.010802,0.026187,0.147977,0.117551,0.066431,0.104967,0.142966,0.146465,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.911140
9,1.159041,0.883567,0.892860,1.029882,1.145041,0.888992,0.967057,0.978419,1.049884,1.107673,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.863876
12,0.157461,0.180588,0.167887,0.071678,0.481999,0.408572,0.154823,0.158678,0.267428,0.202496,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.830230


In [34]:
predict_1 = pd.Series(bestForest_1.predict(scaled_X_test_1), index = scaled_X_test_1.index)

In [35]:
predict_1

0       5.459491
3       4.422107
6       5.645119
9       7.530391
12      6.128772
          ...   
4953    7.190152
4956    5.572607
4959    7.364238
4962    6.922082
4965    4.831595
Length: 1656, dtype: float64

Получили первый прогноз, самое время получать и второй и третий

In [36]:
data_2, columns_2, freeColumn_2 = get_dataFrame_with_params(TwoMonthDataFrame, transformedFillData, [2, 3, 5, 6, 12], 2021, 11, [3, 6, 12])
scaled_X_test_2 = scale_data(data_2, columns_2, freeColumn_2, yearsScaler, scaler_2, monthEncoder, unitEncoder, skuEncoder)
predict_2 = pd.Series(bestForest_2.predict(scaled_X_test_2), index = scaled_X_test_2.index)

In [37]:
data_3, columns_3, freeColumn_3 = get_dataFrame_with_params(ThreeMonthDataFrame, transformedFillData, [3, 4, 5, 12], 2021, 11, [3, 6, 12])
scaled_X_test_3 = scale_data(data_3, columns_3, freeColumn_3, yearsScaler, scaler_3, monthEncoder, unitEncoder, skuEncoder)
predict_3 = pd.Series(bestForest_3.predict(scaled_X_test_3), index = scaled_X_test_3.index)

In [38]:
allPrediction = pd.concat([predict_1, predict_2, predict_3]).sort_index()
allPrediction

0       5.459491
1       5.487418
2       5.505915
3       4.422107
4       4.149595
          ...   
4963    6.607862
4964    6.777015
4965    4.831595
4966    4.717338
4967    4.669370
Length: 4968, dtype: float64

In [39]:
result = ids.copy()
result['prediction'] = allPrediction.round(9)
result['prediction'] = np.expm1(result['prediction'])
result

,id,prediction
0,0_2021-12,233.977840
1,0_2022-01,240.632391
2,0_2022-02,245.143545
3,1_2021-12,82.271549
4,1_2022-01,62.408342
...,...,...
4963,1654_2022-01,739.897115
4964,1654_2022-02,876.445377
4965,1655_2021-12,124.410886
4966,1655_2022-01,110.870060


In [40]:
result.to_csv('data/predict.csv', index=False)